# Исследование влияния миграции

**Цель:** Изучить, как интенсивность миграции между городами влияет
на скорость распространения инфекции.

## Теоретическое введение

Миграция создаёт связи между городами, позволяя инфекции распространяться
за пределы первоначального очага. При высокой миграции:
- Инфекция быстрее достигает новых городов
- Пик заболеваемости наступает раньше
- Может увеличиться общее число заболевших

In [1]:
using DrWatson
@quickactivate

using Agents, DataFrames, Plots, CSV, Random, Statistics
include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

## Функция создания матрицы миграции

In [2]:
function create_migration_matrix(C, intensity)
    M = ones(C, C) .* intensity ./ (C-1)
    for i = 1:C
        M[i, i] = 1 - intensity
    end
    return M
end

create_migration_matrix (generic function with 1 method)

## Функция измерения времени пика

In [3]:
function peak_time(p)
    migration_rates = create_migration_matrix(p[:C], p[:migration_intensity])

    model = initialize_sir(;
        Ns = p[:Ns],
        β_und = p[:β_und],
        β_det = p[:β_det],
        infection_period = p[:infection_period],
        detection_time = p[:detection_time],
        death_rate = p[:death_rate],
        reinfection_probability = p[:reinfection_probability],
        Is = p[:Is],
        seed = p[:seed],
        migration_rates = migration_rates,
    )

    infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    peak = 0.0
    peak_step = 0

    for step = 1:p[:n_steps]
        Agents.step!(model, 1)
        frac = infected_frac(model)
        if frac > peak
            peak = frac
            peak_step = step
        end
    end

    return (peak_time = peak_step, peak_value = peak)
end

peak_time (generic function with 1 method)

## Сканирование интенсивностей

In [4]:
migration_intensities = 0.0:0.1:0.5
seeds = [42, 43, 44]

params_list = []
for mig in migration_intensities
    for s in seeds
        push!(
            params_list,
            Dict(
                :migration_intensity => mig,
                :C => 3,
                :Ns => [1000, 1000, 1000],
                :β_und => [0.5, 0.5, 0.5],
                :β_det => [0.05, 0.05, 0.05],
                :infection_period => 14,
                :detection_time => 7,
                :death_rate => 0.02,
                :reinfection_probability => 0.1,
                :Is => [1, 0, 0],
                :seed => s,
                :n_steps => 150,
            ),
        )
    end
end

println("="^60)
println("ИССЛЕДОВАНИЕ ВЛИЯНИЯ МИГРАЦИИ")
println("="^60)
println("Всего экспериментов: $(length(params_list))")

results = []
for (i, params) in enumerate(params_list)
    data = peak_time(params)
    push!(results, merge(params, Dict(pairs(data))))
    if i % 5 == 0
        println("  Прогресс: $i/$(length(params_list))")
    end
end

ИССЛЕДОВАНИЕ ВЛИЯНИЯ МИГРАЦИИ
Всего экспериментов: 18
  Прогресс: 5/18
  Прогресс: 10/18
  Прогресс: 15/18


## Сохранение и визуализация

In [5]:
df = DataFrame(results)
CSV.write(datadir("migration_scan_all.csv"), df)

grouped = combine(
    groupby(df, [:migration_intensity]),
    :peak_time => mean => :mean_peak_time,
    :peak_value => mean => :mean_peak_value,
)

Row,migration_intensity,mean_peak_time,mean_peak_value
,Float64,Float64,Float64
1,0.0,15.0,0.333259
2,0.1,22.0,1.0
3,0.2,16.0,1.0
4,0.3,17.0,1.0
5,0.4,19.0,1.0
6,0.5,20.0,1.0


График времени пика

In [6]:
plot(grouped.migration_intensity, grouped.mean_peak_time,
     marker = :circle,
     xlabel = "Интенсивность миграции",
     ylabel = "Время до пика (дни)",
     label = "Время пика",
     linewidth = 2,
     color = :blue)
title!("Влияние миграции на время достижения пика")
savefig(plotsdir("migration_peak_time.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab04/project/plots/migration_peak_time.png"

График пиковой заболеваемости

In [7]:
plot(grouped.migration_intensity, grouped.mean_peak_value .* 3000,
     marker = :square,
     xlabel = "Интенсивность миграции",
     ylabel = "Численность в пике",
     label = "Пиковая заболеваемость",
     linewidth = 2,
     color = :red)
title!("Влияние миграции на пик заболеваемости")
savefig(plotsdir("migration_peak_value.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab04/project/plots/migration_peak_value.png"

## Анализ

In [8]:
println("\n" * "="^60)
println("РЕЗУЛЬТАТЫ")
println("="^60)

min_time_row = grouped[argmin(grouped.mean_peak_time), :]
println("\n📊 Время пика от интенсивности:")
for row in eachrow(grouped)
    println("  intensity = $(row.migration_intensity): время = $(round(row.mean_peak_time, digits=1)) дней")
end

println("\n🎯 Оптимальная интенсивность для быстрого распространения:")
println("  Интенсивность = $(min_time_row.migration_intensity)")
println("  Время до пика = $(round(min_time_row.mean_peak_time, digits=1)) дней")

println("\n✅ Исследование миграции завершено!")


РЕЗУЛЬТАТЫ

📊 Время пика от интенсивности:
  intensity = 0.0: время = 15.0 дней
  intensity = 0.1: время = 22.0 дней
  intensity = 0.2: время = 16.0 дней
  intensity = 0.3: время = 17.0 дней
  intensity = 0.4: время = 19.0 дней
  intensity = 0.5: время = 20.0 дней

🎯 Оптимальная интенсивность для быстрого распространения:
  Интенсивность = 0.0
  Время до пика = 15.0 дней

✅ Исследование миграции завершено!
